# Conditional operations through `qm-backend`

This notebook covers **fast reset, T-gate teleportation, quantum state
teleportation and repetition-code feedback** through `qm-backend`. You build and draw the
circuit locally, then submit that same construction code to the server over SSH.
There is no separately maintained server script.

The notebook kernel needs the CQT-qibo fork with conditional operations, IPython,
pandas and matplotlib. The QM stack runs in the server's isolated
`test-qm-backend/.venv`, using the `eo-qm-backend` CQT-qibo fork installed there.
The local kernel does not load the hardware backend or execute the circuit.

Run the setup and circuit cells, then the explicit submission cell only when you
want to launch a real job. There is no simulation or automatic fallback.
All four experiments use the same submission and retrieval mechanism.

In [9]:
# Local notebook tools. Circuit construction also uses the local CQT-qibo fork.
import json
import re
import shlex
import subprocess
import time
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.core.magic import register_cell_magic


@register_cell_magic
def server_code(line, cell):
    """Store remote-only Python source without executing it locally."""
    get_ipython().user_ns[line.strip()] = cell
    print(f"Stored {line.strip()}; nothing executed or submitted.")


@register_cell_magic
def circuit_code(line, cell):
    """Build the circuit locally and retain the exact same source for Slurm."""
    namespace = get_ipython().user_ns
    name = line.strip()
    namespace.pop(name, None)
    exec(compile(cell, "<notebook circuit construction>", "exec"), namespace)
    namespace[name] = cell
    print(f"Circuit built locally; source stored as {name}. No job submitted.")

In [10]:
# SSH / Slurm transport. Only submit_notebook_job() submits a job.
SSH_HOST = "cqt-paul-server-nqch-deploy"
REMOTE_DIR = "/mnt/home/nqch-deploy/test-qm-backend"
REMOTE_PYTHON = f"{REMOTE_DIR}/.venv/bin/python"
PLATFORMS_ROOT = "/mnt/scratch/qibolab_platforms_nqch"
SLURM_LOGS = f"{REMOTE_DIR}/slurm_logs"
RESULT_PREFIX = "QM_BACKEND_RESULT="
TERMINAL_STATES = {
    "COMPLETED", "FAILED", "CANCELLED", "TIMEOUT", "OUT_OF_MEMORY",
    "NODE_FAIL", "BOOT_FAIL", "DEADLINE", "PREEMPTED", "REVOKED",
}


def _ssh(command, *, input_text=None):
    completed = subprocess.run(
        ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=10", SSH_HOST, command],
        input=input_text if input_text is not None else "",
        capture_output=True, text=True, timeout=45,
    )
    if completed.returncode:
        raise RuntimeError(f"SSH command failed: {completed.stderr.strip()}")
    return completed.stdout


def _slurm_state(job_id):
    rows = _ssh(f"sacct -X -n -P -j {shlex.quote(str(job_id))} --format=JobIDRaw,State")
    for row in rows.splitlines():
        fields = row.split("|", 1)
        if len(fields) == 2 and fields[0].strip() == str(job_id):
            return fields[1].strip().split()[0].rstrip("+")
    queued = _ssh(f"squeue -h -j {shlex.quote(str(job_id))} -o %T").strip()
    return queued.splitlines()[0] if queued else "UNKNOWN"


def submit_notebook_job(source, log_name="fast-reset"):
    """Send the stored cell verbatim to sbatch's stdin; no server .py file."""
    delimiter = "NOTEBOOK_PYTHON_" + uuid.uuid4().hex
    batch = (
        "#!/bin/bash\n"
        f"export QM_BACKEND_PLATFORMS={shlex.quote(PLATFORMS_ROOT)}\n"
        f"exec {shlex.quote(REMOTE_PYTHON)} -u - <<'{delimiter}'\n"
        + source + ("" if source.endswith("\n") else "\n")
        + delimiter + "\n"
    )
    command = shlex.join([
        "sbatch", "--parsable", "--partition=sinq20", f"--job-name=qm_notebook_{log_name}",
        f"--chdir={REMOTE_DIR}", "--time=00:30:00",
        f"--output={SLURM_LOGS}/slurm-notebook-{log_name}-%j.out",
        f"--error={SLURM_LOGS}/slurm-notebook-{log_name}-%j.err",
    ])
    # Slurm retains the submitted program internally. Re-editing the notebook
    # after submission does not change the running job.
    job_id = _ssh(command, input_text=batch).strip().split(";", 1)[0]
    if not re.fullmatch(r"\d+", job_id):
        raise RuntimeError(f"Could not parse Slurm job ID: {job_id!r}")
    print(f"Submitted Slurm job {job_id}", flush=True)
    return job_id

## Fast reset — build and draw the circuit locally

Prepare $|+\rangle$, measure and conditionally apply X, then repeat the reset
before the final measurement. On the server, logical qubit 0 will use physical
qubit **q5**, with **1,000 shots**.

`%%circuit_code fast_reset_circuit_source` runs the construction cell locally,
so `c` and its measurement handles are available in the notebook. It also stores
that exact code for submission. This cell contains imports, circuit
construction and `c.draw()`, so running it also displays the circuit immediately.
Backend selection and `c(nshots=...)` remain in the remote cells below. The
stored code includes `c.draw()`, so the drawing also appears in the Slurm output.

Edit and rerun the construction cell whenever you change the circuit. The local
drawing and the submitted construction code then reflect the same definition.
Changes made to `c` in other cells are local only; put gate edits in this cell.
The result and plotting cells identify measurements by their `register_name`
values (`before_reset`, `before_reset2`, `after_reset`). Renaming the Python
measurement variables does not change those results or require plotting changes.


In [11]:
%%circuit_code fast_reset_circuit_source
from qibo import Circuit, gates

c = Circuit(1)
c.add(gates.H(0))  # prepare |+> so both feedback outcomes occur

measured_bit_before_reset_1 = c.add(gates.M(0, collapse=True, register_name="before_reset"))
c.add(gates.IfElse(
    measured_bit_before_reset_1.is_one(), 
    then_do=gates.X(0)))

measured_bit_before_reset2 = c.add(gates.M(0, collapse=True, register_name="before_reset2"))
c.add(gates.IfElse(
    measured_bit_before_reset2.is_one(), 
    then_do=gates.X(0)))
c.add(gates.M(0, register_name="after_reset"))

c.draw()


0: ─H─M─if[─X─][─]─M─if[─X─][─]─M─
Circuit built locally; source stored as fast_reset_circuit_source. No job submitted.


### Hardware setup and execution — stored for the server only

The next two cells store the server setup and execution code without running
it on your Mac. At submission, the notebook joins these pieces in this order:

1. Select `qm-backend`, platform `sinq20`, and physical placement q5.
2. Build the circuit using the exact construction cell above.
3. Run it for 1,000 shots and write its results to the Slurm output.

After a successful hardware run, the code writes the measurement counts as JSON
text in the `.out` file. The notebook retrieves those counts over SSH for plotting.

In [12]:
%%server_code fast_reset_setup_source
import json
import qibo

qibo.set_backend("qm-backend",
                platform="sinq20",
                physical_qubits=("q5",))

Stored fast_reset_setup_source; nothing executed or submitted.


In [13]:
%%server_code fast_reset_execution_source

result = c(nshots=1000)
print(f"QM job ID: {result.qm_job_id}", flush=True)
print("\nFinal counts:", result.frequencies(), flush=True)
print("\nAll measurement registers:", result.register_frequencies(), flush=True)

# Preserve shot correlations, with register order stated explicitly.
registers = ("before_reset", "before_reset2", "after_reset")
payload = {
    "qm_job_id": result.qm_job_id,
    "shots": result.nshots,
    "register_counts": {name: dict(counts) for name, counts in result.register_frequencies().items()},
    "joint_counts": dict(result.joint_frequencies(registers=registers)),
    "final_counts": dict(result.frequencies()),
}
print("QM_BACKEND_RESULT=" + json.dumps(payload), flush=True)

Stored fast_reset_execution_source; nothing executed or submitted.


## Retrieve results and logs

These functions wait for the submitted job and retrieve its `.out` and `.err`
files from `slurm_logs/`. Measurement data comes from the execution cell's JSON
line, using Qibo register order. The adapter below supplies the same counts
expected by the existing plots.

A result payload is read whenever it is present, independently of the Slurm
status. The actual Slurm status is still reported. If no payload is present,
`measurement_results` is `None`; inspect `raw_logs`. A successful Slurm exit
without a payload is reported as a retrieval error.

In [14]:
def _fast_reset_measurements(payload):
    registers = payload["register_counts"]
    before = registers["before_reset"]
    before2 = registers["before_reset2"]
    after = registers["after_reset"]
    return {
        "joint_counts": {"->".join(key.split()): count for key, count in payload["joint_counts"].items()},
        "before_reset_counts": before,
        "before_reset2_counts": before2,
        "after_reset_counts": after,
        "first_reset_zero_count": before2.get("0", 0),
        "final_zero_count": after.get("0", 0),
    }


def collect_notebook_job(job_id, log_name="fast-reset"):
    """Retrieve a job again without submitting anything; useful after interruption."""
    job_id = str(job_id)
    status = _slurm_state(job_id)
    log_base = f"{SLURM_LOGS}/slurm-notebook-{log_name}-{job_id}"
    # Slurm may record completion just before the files become visible.
    for attempt in range(5):
        try:
            stdout = _ssh(f"cat {shlex.quote(log_base + '.out')}")
            stderr = _ssh(f"cat {shlex.quote(log_base + '.err')}")
            break
        except RuntimeError:
            if attempt == 4:
                raise
            time.sleep(2)
    records = [line[len(RESULT_PREFIX):] for line in stdout.splitlines() if line.startswith(RESULT_PREFIX)]
    payload = json.loads(records[-1]) if records else None
    retrieval_error = None
    if status == "COMPLETED" and payload is None:
        retrieval_error = "Job completed but no backend result payload was found in stdout."
    return {
        "job_id": job_id, "status": status,
        "raw_logs": {"stdout": stdout, "stderr": stderr},
        "backend_result": payload,
        "measurement_results": payload,
        "retrieval_error": retrieval_error,
    }


def wait_for_notebook_job(job_id, log_name="fast-reset", poll_seconds=10, max_wait_seconds=7200):
    """Wait only: this function never submits or cancels a job."""
    started = time.monotonic()
    previous_state = None
    while True:
        status = _slurm_state(job_id)
        if status != previous_state:
            print(f"Job {job_id}: {status}", flush=True)
            previous_state = status
        if status in TERMINAL_STATES:
            return collect_notebook_job(job_id, log_name=log_name)
        if time.monotonic() - started >= max_wait_seconds:
            raise TimeoutError(f"Stopped waiting for job {job_id}; it was not cancelled. Use collect_notebook_job({job_id!r}, log_name={log_name!r}) later.")
        time.sleep(poll_seconds)

def collect_fast_reset_job(job_id):
    job = collect_notebook_job(job_id)
    payload = job["backend_result"]
    job["measurement_results"] = _fast_reset_measurements(payload) if payload is not None else None
    return job


def wait_for_fast_reset_job(job_id, poll_seconds=10, max_wait_seconds=7200):
    job = wait_for_notebook_job(job_id, poll_seconds=poll_seconds, max_wait_seconds=max_wait_seconds)
    payload = job["backend_result"]
    job["measurement_results"] = _fast_reset_measurements(payload) if payload is not None else None
    return job


## Submit — running the next cell launches a real hardware job

The next cell combines the stored backend setup, circuit construction and
execution code into `fast_reset_source`, then sends that program to Slurm over
SSH using the server `.venv` and `sinq20` partition. Slurm log files (`.out` and
`.err`) are written in `slurm_logs/`.

Run the construction and remote-code cells after editing them, then submit
**once per desired job**. Its job ID is saved before waiting. If you interrupt
the wait, the job continues; retrieve it later with
`fast_reset_job = collect_fast_reset_job(fast_reset_job_id)` instead of resubmitting.

In [15]:
# Submit slurm job
fast_reset_source = "\n\n".join((
    fast_reset_setup_source,
    fast_reset_circuit_source,
    fast_reset_execution_source,
))
fast_reset_job_id = submit_notebook_job(fast_reset_source)
fast_reset_job = wait_for_fast_reset_job(fast_reset_job_id)
print("Slurm state:", fast_reset_job["status"])
if fast_reset_job["measurement_results"] is None:
    print(fast_reset_job["retrieval_error"] or "No hardware results returned.")
    print("\n--- stdout ---\n" + fast_reset_job["raw_logs"]["stdout"])
    print("\n--- stderr ---\n" + fast_reset_job["raw_logs"]["stderr"])
fast_reset_job["measurement_results"]

Submitted Slurm job 67594
Job 67594: PENDING
Job 67594: RUNNING
Job 67594: FAILED
Slurm state: FAILED
No hardware results returned.

--- stdout ---
2026-11-06 22:30:27,870 - qm - INFO     - Starting session: 6bb820e8-7702-469a-8e0f-8765b12f3091
[Qibo 0.3.7|INFO|2026-11-06 22:30:29]: Using qm-backend (sinq20) backend on /CPU:0
0: ─H─M─if[─X─][─]─M─if[─X─][─]─M─
2026-11-06 22:32:29,696 - qm - ERROR    - Failed to detect to QuantumMachines server, failed to connect to cluster 'Incredible_Horse'. Tried connecting to 192.168.88.246:80.
Errors:
192.168.88.246:80: timed out
Traceback (most recent call last):
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

In [16]:
#fast_reset_job = collect_fast_reset_job("67592")

In [17]:
# Plotting
results = fast_reset_job["measurement_results"]
if results is None:
    raise ValueError("No measurement results found; inspect fast_reset_job['raw_logs'].")

transitions = ["->".join(f"{value:03b}") for value in range(8)]  # before_reset1 -> before_reset2 -> after_reset
joint_counts = pd.DataFrame(
    {"Joint counts": [results["joint_counts"].get(label, 0) for label in transitions]},
    index=transitions,
)
before_after_counts = pd.DataFrame(
    {
        "Before reset1": [results["before_reset_counts"].get(bit, 0) for bit in ("0", "1")],
        "Before reset2": [results["before_reset2_counts"].get(bit, 0) for bit in ("0", "1")],
        "After reset": [results["after_reset_counts"].get(bit, 0) for bit in ("0", "1")],
    },
    index=["0", "1"],
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
shots = int(joint_counts["Joint counts"].sum())
succ_rate = results['final_zero_count']/shots
fig.suptitle(f"Fast reset: success rate = {results['final_zero_count']}/{shots}={succ_rate}")
joint_counts.plot.bar(ax=axes[0], color="tab:blue", rot=45, legend=True)
before_after_counts.plot.bar(
    ax=axes[1], color=["tab:purple", "tab:green", "tab:orange"], rot=0, legend=True
)
axes[0].set(title="Joint outcomes", xlabel="before1 -> before2 -> after", ylabel="Shots")
axes[1].set(title="Before and after each reset", xlabel="Measured value", ylabel="Shots")
for ax in axes:
    for bars in ax.containers:
        ax.bar_label(bars, padding=3)
    ax.margins(y=0.15)
plt.show()

ValueError: No measurement results found; inspect fast_reset_job['raw_logs'].

**How to read the fast-reset plots**

- **Left (blue):** Each bar counts a three-measurement sequence from the same
  shot: before reset 1 → before reset 2 → after reset 2. For example, `1→0→0`
  means the first correction left 0, which remained 0 after the second round.
- **Right:** Purple shows the first measurement, green the second, and orange
  the final measurement. A taller orange bar at 0 means more successful final
  resets. The title reports the final-zero count divided by all returned shots.

These plots require successful hardware results retrieved above. The notebook
contains no saved replacement results for the currently unreachable QM server.

## T-gate teleportation

Qubit 0 holds the input $|+\rangle$ and qubit 1 the magic state $T|+\rangle$. After the
CNOT, measuring qubit 1 leaves $T|+\rangle$ on qubit 0 for outcome 0 and $T^\dagger|+\rangle$
for outcome 1, which the conditional $S$ corrects ($S\,T^\dagger = T$). Measuring
$T|+\rangle$ in the X basis gives $P(1) = \sin^2(\pi/8) \approx 0.146$.
Logical qubits 0, 1 and 2 use **q15, q19 and q5**, respectively, with
**10,000 shots**. The independent q5 circuit applies T directly as a reference.
The measured probability tests this input and measurement basis; it is not a
full gate-fidelity measurement.

In [18]:
%%circuit_code teleportation_circuit_source
from qibo import Circuit, gates

c_tgate = Circuit(3)

# Prepare state |+⟩ for q_0
c_tgate.add(gates.H(0)) 

# Prepare state T|+⟩ for q_1
c_tgate.add(gates.H(1)) 
c_tgate.add(gates.T(1)) # here T gate to add a 45º phase

# CNOT Links the two qubits so the T phase is part of their shared state.
c_tgate.add(gates.CNOT(0, 1)) 

# Then we measure qubit 1 because:
# if q_1= 0 then q_0 will be in T|+⟩ state so no need to do anything. We've teleported the phase.
# if q_1 =1 then q_0 will be in Tdagga|+⟩ state (the phase is in the |0⟩ not the |1⟩)
# in that case we need to apply an S gate to correct and leave q_0 in T|+⟩
m1 = c_tgate.add(gates.M(1, collapse=True, register_name="feedback"))
c_tgate.add(gates.IfElse(m1.is_one(), then_do=gates.S(0)))

# Measure qubit_0 to get how well we teleported the T-gate
# measurement is H+M coz quantum measurements can only see probabilities (amplitudes), not phases.
c_tgate.add(gates.H(0))
m0 = c_tgate.add(gates.M(0, register_name="teleported"))

# # Prepare state T|+⟩ for q_2 & measure to have reference to compare our teleportation against
c_tgate.add(gates.H(2))
c_tgate.add(gates.T(2))
c_tgate.add(gates.H(2))
m2 = c_tgate.add(gates.M(2, register_name="local"))

c_tgate.draw()

0: ─H───o───if[─S─][─]─H─M─
1: ─H─T─X─M────────────────
2: ───────H─T───H─M────────
Circuit built locally; source stored as teleportation_circuit_source. No job submitted.


In [19]:
%%server_code teleportation_setup_source
import json
import qibo

qibo.set_backend("qm-backend", platform="sinq20", physical_qubits=('q15', 'q19', 'q5'))

Stored teleportation_setup_source; nothing executed or submitted.


In [20]:
%%server_code teleportation_execution_source
result = c_tgate(nshots=10000)
print(f"QM job ID: {result.qm_job_id}", flush=True)
print("\nFinal counts:", result.frequencies(), flush=True)
print("\nAll measurement registers:", result.register_frequencies(), flush=True)

payload = {
    "qm_job_id": result.qm_job_id,
    "shots": result.nshots,
    "register_counts": {name: dict(counts) for name, counts in result.register_frequencies().items()},
    "joint_counts": dict(result.joint_frequencies(registers=('feedback', 'teleported', 'local'))),
    "final_counts": dict(result.frequencies()),
}
print("QM_BACKEND_RESULT=" + json.dumps(payload), flush=True)

Stored teleportation_execution_source; nothing executed or submitted.


In [21]:
# Submit one job.
teleportation_source = "\n\n".join((
    teleportation_setup_source,
    teleportation_circuit_source,
    teleportation_execution_source,
))
teleportation_job_id = submit_notebook_job(teleportation_source, log_name='teleportation')
teleportation_job = wait_for_notebook_job(teleportation_job_id, log_name='teleportation')
print("Slurm state:", teleportation_job["status"])
if teleportation_job["measurement_results"] is None:
    print(teleportation_job["retrieval_error"] or "No hardware results returned.")
    print("\n--- stdout ---\n" + teleportation_job["raw_logs"]["stdout"])
    print("\n--- stderr ---\n" + teleportation_job["raw_logs"]["stderr"])
teleportation_job["measurement_results"]

Submitted Slurm job 67595
Job 67595: RUNNING
Job 67595: FAILED
Slurm state: FAILED
No hardware results returned.

--- stdout ---
2026-11-06 22:34:29,869 - qm - INFO     - Starting session: b711d8ff-a0f3-48e2-8c37-27de67462f68
[Qibo 0.3.7|INFO|2026-11-06 22:34:31]: Using qm-backend (sinq20) backend on /CPU:0
0: ─H───o───if[─S─][─]─H─M─
1: ─H─T─X─M────────────────
2: ───────H─T───H─M────────
2026-11-06 22:36:31,687 - qm - ERROR    - Failed to detect to QuantumMachines server, failed to connect to cluster 'Incredible_Horse'. Tried connecting to 192.168.88.246:80.
Errors:
192.168.88.246:80: timed out
Traceback (most recent call last):
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^

In [22]:
# Teleportation plots
results = teleportation_job["measurement_results"]
if results is None:
    raise ValueError("No measurement results found; inspect teleportation_job['raw_logs'].")

shots = results["shots"]
register_counts = results["register_counts"]
results = {
    "shots": shots,
    "p1_teleported": register_counts["teleported"].get("1", 0) / shots,
    "p1_local": register_counts["local"].get("1", 0) / shots,
}

p1_values = pd.Series(
    {
        "Teleported (q0)": results["p1_teleported"],
        "Local HTH (q2)": results["p1_local"],
    }
)
theoretical_p1 = np.sin(np.pi / 8) ** 2  # Approximately 0.146
relative_error_percent = (p1_values - theoretical_p1).abs() / theoretical_p1 * 100
# Binomial shot-noise standard error of each P(1) estimate.
p1_sigma = np.sqrt(p1_values * (1 - p1_values) / results["shots"])
deviation_sigma = (p1_values - theoretical_p1).abs() / p1_sigma

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
fig.suptitle(
    f"P(1)_rel-err: teleported = {relative_error_percent['Teleported (q0)']:.1f}% "
    f"({deviation_sigma['Teleported (q0)']:.1f}σ), "
    f"local = {relative_error_percent['Local HTH (q2)']:.1f}% "
    f"({deviation_sigma['Local HTH (q2)']:.1f}σ)"
)
p1_values.plot.bar(ax=ax, color=["tab:pink", "tab:cyan"], rot=0)
ax.errorbar(
    range(len(p1_values)), p1_values, yerr=3 * p1_sigma,
    fmt="none", ecolor="black", capsize=8, label="±3σ shot noise",
)
ax.axhline(
    theoretical_p1, color="black", linestyle="--",
    label=f"Target sin²(π/8) = {theoretical_p1:.3f}",
)
ax.set(title=f"P(1) over {results['shots']} shots", ylabel="P(1)")
ax.bar_label(ax.containers[0], fmt="%.3f", label_type="center", color="white")
ax.legend()
ax.margins(y=0.15)
plt.show()

ValueError: No measurement results found; inspect teleportation_job['raw_logs'].

The pink and cyan bars compare the teleported and directly applied T phase.
The dashed line marks $\sin^2(\pi/8)$; error bars are $\pm3$ binomial standard
errors estimated from each measured probability. They describe shot noise,
not calibration uncertainty. The title reports deviation from the ideal target.

## Quantum state teleportation

Three-qubit teleportation of the state $|-\rangle$. Qubit 0 holds the input $|-\rangle$, prepared with $X$ then $H$; qubits 1 and 2 form an entangled resource using Hadamards and a CZ. A CZ-based Bell measurement of qubits 0 and 1 gives `feedback_z` (qubit 0) and `feedback_x` (qubit 1), which conditionally apply $Z$ and $X$ to qubit 2. A final $H$ maps the teleported $|-\rangle$ to $|1\rangle$, so measuring `result` = 1 counts as a successful teleportation.

On hardware, logical qubits 0, 1, 2 are **q19, q15, q10**; the two CZ gates use the q15–q10 and q19–q15 couplings.
Use **1,000 shots**. A final outcome of 1 is the expected result for this
particular prepared state and analysis basis; its fraction alone does not
establish arbitrary-state teleportation fidelity.

In [23]:
%%circuit_code quantum_teleportation_circuit_source
from qibo import Circuit, gates

c_quantum = Circuit(3)

# Prepare state |-⟩ for q_0, this is the state to teleport
c_quantum.add(gates.X(0))
c_quantum.add(gates.H(0))  

# Prepare Bell state of q_1 & q_2 so they are maximally entangled
# CZ gate achieves so by flipping the sign of |11>
c_quantum.add(gates.H(2))
c_quantum.add(gates.H(1))
c_quantum.add(gates.CZ(1, 2))  

# Entangle qubits q_0 & q_1 so their states are correlated
c_quantum.add(gates.CZ(0, 1))

# Apply Hadamard gates to qubits q_0 and q_1 to rotate them back to the Z-axis measurement basis
c_quantum.add(gates.H(1))
c_quantum.add(gates.H(0))

# Measure qubits q_1 and q_0 to obtain classical feedback bits
feedback_x = c_quantum.add(gates.M(1, collapse=True, register_name="feedback_x"))
feedback_z = c_quantum.add(gates.M(0, collapse=True, register_name="feedback_z"))

# Conditionally correct qubit 2 based on the measurement feedback:
# If feedback_x is 1, apply an X gate (bit-flip) to qubit 2
# If feedback_z is 1, apply a Z gate (phase-flip) to qubit 2
c_quantum.add(gates.IfElse(feedback_x.is_one(), then_do=gates.X(2)))
c_quantum.add(gates.IfElse(feedback_z.is_one(), then_do=gates.Z(2)))

# Apply a final Hadamard to qubit 2 to map the teleported |-⟩ state to |1⟩ for measurement
c_quantum.add(gates.H(2))  
c_quantum.add(gates.M(2, register_name="result"))

# The key here in this teleportation is that 
# collapsing qubits q_0 and q_1 leaves qubit q_2 holding the exact original
# |-⟩ state (plus or minus an X or Z flip) because of the entanglement.

c_quantum.draw()

0: ─X─H───o─H─M───────────────────────────
1: ─H───o─Z─H─M───────────────────────────
2: ─H───Z───────if[─X─][─]─if[─Z─][─]─H─M─
Circuit built locally; source stored as quantum_teleportation_circuit_source. No job submitted.


In [24]:
%%server_code quantum_teleportation_setup_source
import json
import qibo

qibo.set_backend("qm-backend", platform="sinq20", physical_qubits=('q19', 'q15', 'q10'))

Stored quantum_teleportation_setup_source; nothing executed or submitted.


In [25]:
%%server_code quantum_teleportation_execution_source
result = c_quantum(nshots=1000)
print(f"QM job ID: {result.qm_job_id}", flush=True)
print("\nFinal counts:", result.frequencies(), flush=True)
print("\nAll measurement registers:", result.register_frequencies(), flush=True)

payload = {
    "qm_job_id": result.qm_job_id,
    "shots": result.nshots,
    "register_counts": {name: dict(counts) for name, counts in result.register_frequencies().items()},
    "joint_counts": dict(result.joint_frequencies(registers=('feedback_x', 'feedback_z', 'result'))),
    "final_counts": dict(result.frequencies()),
}
print("QM_BACKEND_RESULT=" + json.dumps(payload), flush=True)

Stored quantum_teleportation_execution_source; nothing executed or submitted.


In [26]:
# Submit one job.
quantum_teleportation_source = "\n\n".join((
    quantum_teleportation_setup_source,
    quantum_teleportation_circuit_source,
    quantum_teleportation_execution_source,
))
quantum_teleportation_job_id = submit_notebook_job(quantum_teleportation_source, log_name='quantum-teleportation')
quantum_teleportation_job = wait_for_notebook_job(quantum_teleportation_job_id, log_name='quantum-teleportation')
print("Slurm state:", quantum_teleportation_job["status"])
if quantum_teleportation_job["measurement_results"] is None:
    print(quantum_teleportation_job["retrieval_error"] or "No hardware results returned.")
    print("\n--- stdout ---\n" + quantum_teleportation_job["raw_logs"]["stdout"])
    print("\n--- stderr ---\n" + quantum_teleportation_job["raw_logs"]["stderr"])
quantum_teleportation_job["measurement_results"]

Submitted Slurm job 67596
Job 67596: RUNNING
Job 67596: FAILED
Slurm state: FAILED
No hardware results returned.

--- stdout ---
2026-11-06 22:38:58,861 - qm - INFO     - Starting session: f285005d-1229-425d-b694-c04ce481e0cc
[Qibo 0.3.7|INFO|2026-11-06 22:39:00]: Using qm-backend (sinq20) backend on /CPU:0
0: ─X─H───o─H─M───────────────────────────
1: ─H───o─Z─H─M───────────────────────────
2: ─H───Z───────if[─X─][─]─if[─Z─][─]─H─M─
2026-11-06 22:41:00,629 - qm - ERROR    - Failed to detect to QuantumMachines server, failed to connect to cluster 'Incredible_Horse'. Tried connecting to 192.168.88.246:80.
Errors:
192.168.88.246:80: timed out
Traceback (most recent call last):
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._

In [27]:
# Quantum teleportation plot
from matplotlib.patches import Patch

results = quantum_teleportation_job["measurement_results"]
if results is None:
    raise ValueError("No measurement results found; inspect quantum_teleportation_job['raw_logs'].")

counts = results["register_counts"]["result"]
shots = results["shots"]
results = {
    "result_counts": counts, "shots": shots,
    "final_one_count": counts.get("1", 0),
    "p1": counts.get("1", 0) / shots,
}

result_labels = ["0", "1"]
result_counts = pd.Series(
    {label: results["result_counts"].get(label, 0) for label in result_labels}
)

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
fig.suptitle(
    f"Quantum teleportation: success = {results['final_one_count']}/{results['shots']} "
    f"({results['p1']:.1%})"
)
result_colors = ["tab:green" if label == "1" else "tab:orange" for label in result_labels]
result_counts.plot.bar(ax=ax, color=result_colors, rot=0)
ax.set(title="Result counts", xlabel="Result (q10)", ylabel="Shots")
ax.legend(handles=[
    Patch(color="tab:orange", label="Failure: 0"),
    Patch(color="tab:green", label="Success: 1"),
])
for bars in ax.containers:
    ax.bar_label(bars, padding=3)
ax.margins(y=0.15)
plt.show()

ValueError: No measurement results found; inspect quantum_teleportation_job['raw_logs'].

Green counts the expected final outcome 1 on q10; orange counts outcome 0.
The title gives the observed fraction of 1 outcomes after measurement-dependent
X and Z corrections, followed by the final H analysis gate.

## Repetition code

Three data qubits encode **one logical qubit**; two ancillas read its parity checks. In a repetition code, parity checks let you detect if an error (like an unintended bit-flip) has occurred. This example prepares every data qubit in $|+\rangle$, so the parity measurement gives each of the four syndromes with equal probability, and uses `Switch` to correct each one. A syndrome is the combined result of all your parity check measurements. After correction the data should read only `000` or `111`, about half the shots each.

The circuit wires 0–4 occupy **q4, q5, q10, q15, q19**, which form a line on the chip: data q4, ancilla q5, data q10, ancilla q15, data q19. Each ancilla is coupled to both data qubits it checks (q5 to q4 and q10; q15 to q10 and q19), so no SWAP is needed. The syndrome is measured in **q5, q15** order; the final data are measured in **q4, q10, q19** order.
Use **1,000 shots**. For syndrome `10`, flip data qubit q4; for `11`, flip
q10; for `01`, flip q19; for `00`, leave the data unchanged. Ideally each
syndrome-conditioned data state is corrected to
$ (|000\rangle + |111\rangle)/\sqrt{2} $.
Finite-shot counts fluctuate around the equal ideal probabilities.
This demonstrates parity measurement and feedback; final populations alone do
not demonstrate preserved coherence or logical error suppression.

In [28]:
%%circuit_code repetition_circuit_source
from qibo import Circuit, gates

# Circuit wire -> physical qubit:
# 0 -> q4 (data 0), 1 -> q5 (ancilla 0), 2 -> q10 (data 1),
# 3 -> q15 (ancilla 1), 4 -> q19 (data 2).
data0, anc0, data1, anc1, data2 = range(5)
c_repetition = Circuit(5, wire_names=["q4", "q5", "q10", "q15", "q19"])

# Earlier version: inject an X error on the middle data state (disabled).
# c_repetition.add(gates.X(data1))

# Put every data qubit in |+> so all four syndromes occur.
c_repetition.add(gates.H(data0))
c_repetition.add(gates.H(data1))
c_repetition.add(gates.H(data2))

c_repetition.add(gates.H(anc0))
c_repetition.add(gates.H(anc1))

# Parity checks let you detect if an error (like an unintended bit-flip) has occurred.
# Parity checks on the q4-q5-q10-q15-q19 line: q5 checks q4/q10, q15 checks q10/q19.
c_repetition.add(gates.CZ(anc0, data1))
c_repetition.add(gates.CZ(anc1, data2))
c_repetition.add(gates.CZ(anc0, data0))
c_repetition.add(gates.CZ(anc1, data1))

c_repetition.add(gates.H(anc0))
c_repetition.add(gates.H(anc1))

syndrome = c_repetition.add(
    gates.M(anc0, anc1, collapse=True, register_name="syndrome")
)
c_repetition.add(
    gates.Switch(syndrome)
    .case("10", gates.X(data0))
    .case("11", gates.X(data1))
    .case("01", gates.X(data2))
    .default([])  # "00": no error
)
c_repetition.add(gates.M(data0, data1, data2, register_name="result"))

c_repetition.draw()

q4 : ─H─────Z───────switch[─X─][───][───][─]─M─
q5 : ─H─o───o───H─M─|─────────|────|────|──|─|─
q10: ─H─Z─────Z───|─switch[───][─X─][───][─]─M─
q15: ─H───o───o─H─M─|─────────|────|────|──|─|─
q19: ─H───Z─────────switch[───][───][─X─][─]─M─
Circuit built locally; source stored as repetition_circuit_source. No job submitted.


In [29]:
%%server_code repetition_setup_source
import json
import qibo

qibo.set_backend("qm-backend", platform="sinq20", physical_qubits=('q4', 'q5', 'q10', 'q15', 'q19'))

Stored repetition_setup_source; nothing executed or submitted.


In [30]:
%%server_code repetition_execution_source
result = c_repetition(nshots=1000)
print(f"QM job ID: {result.qm_job_id}", flush=True)
print("\nFinal counts:", result.frequencies(), flush=True)
print("\nAll measurement registers:", result.register_frequencies(), flush=True)

payload = {
    "qm_job_id": result.qm_job_id,
    "shots": result.nshots,
    "register_counts": {name: dict(counts) for name, counts in result.register_frequencies().items()},
    "joint_counts": dict(result.joint_frequencies(registers=('syndrome', 'result'))),
    "final_counts": dict(result.frequencies()),
}
print("QM_BACKEND_RESULT=" + json.dumps(payload), flush=True)

Stored repetition_execution_source; nothing executed or submitted.


In [31]:
# Submit one job.
repetition_source = "\n\n".join((
    repetition_setup_source,
    repetition_circuit_source,
    repetition_execution_source,
))
repetition_job_id = submit_notebook_job(repetition_source, log_name='repetition')
repetition_job = wait_for_notebook_job(repetition_job_id, log_name='repetition')
print("Slurm state:", repetition_job["status"])
if repetition_job["measurement_results"] is None:
    print(repetition_job["retrieval_error"] or "No hardware results returned.")
    print("\n--- stdout ---\n" + repetition_job["raw_logs"]["stdout"])
    print("\n--- stderr ---\n" + repetition_job["raw_logs"]["stderr"])
repetition_job["measurement_results"]

Submitted Slurm job 67597
Job 67597: PENDING
Job 67597: RUNNING
Job 67597: FAILED
Slurm state: FAILED
No hardware results returned.

--- stdout ---
2026-11-06 22:42:18,857 - qm - INFO     - Starting session: 49559ef6-4974-4ed6-b88f-55ea098f0a44
[Qibo 0.3.7|INFO|2026-11-06 22:42:20]: Using qm-backend (sinq20) backend on /CPU:0
q4 : ─H─────Z───────switch[─X─][───][───][─]─M─
q5 : ─H─o───o───H─M─|─────────|────|────|──|─|─
q10: ─H─Z─────Z───|─switch[───][─X─][───][─]─M─
q15: ─H───o───o─H─M─|─────────|────|────|──|─|─
q19: ─H───Z─────────switch[───][───][─X─][─]─M─
2026-11-06 22:44:20,766 - qm - ERROR    - Failed to detect to QuantumMachines server, failed to connect to cluster 'Incredible_Horse'. Tried connecting to 192.168.88.246:80.
Errors:
192.168.88.246:80: timed out
Traceback (most recent call last):
  File "/mnt/home/nqch-deploy/test-qm-backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/mnt/home/nqch-depl

In [32]:
# Plot Repetition-code syndrome and final-data distributions
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

results = repetition_job["measurement_results"]
if results is None:
    raise ValueError("No measurement results found; inspect repetition_job['raw_logs'].")

results = {
    "shots": results["shots"],
    "syndrome_counts": results["register_counts"]["syndrome"],
    "data_result_counts": results["register_counts"]["result"],
    "joint_counts": {"->".join(key.split()): count for key, count in results["joint_counts"].items()},
}

syndrome_labels = ["00", "01", "10", "11"]
data_labels = [f"{value:03b}" for value in range(8)]
expected_data = ("000", "111")  # every syndrome is corrected to one of these
syndrome_counts = pd.Series(
    {label: results["syndrome_counts"].get(label, 0) for label in syndrome_labels}
)
data_result_counts = pd.Series(
    {label: results["data_result_counts"].get(label, 0) for label in data_labels}
)
codeword_count = int(data_result_counts[list(expected_data)].sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig.suptitle(
    f"Repetition code: final 000 or 111 = {codeword_count}/{results['shots']} "
    f"({codeword_count / results['shots']:.1%})"
)
data_colors = ["tab:green" if label in expected_data else "tab:orange" for label in data_labels]
syndrome_counts.plot.bar(ax=axes[0], color="tab:blue", rot=0)
axes[0].axhline(results["shots"] / len(syndrome_labels), color="black", linestyle="--")
data_result_counts.plot.bar(ax=axes[1], color=data_colors, rot=0)
axes[0].set(title="Syndrome counts", xlabel="Syndrome (ancillas q5, q15)", ylabel="Shots")
axes[1].set(title="Final data counts", xlabel="Final data (q4, q10, q19)", ylabel="Shots")
axes[0].legend(handles=[
    Patch(color="tab:blue", label="Syndrome"),
    Line2D([], [], color="black", linestyle="--", label="Ideal: shots/4 each"),
], loc="upper center", ncols=2)
axes[1].legend(handles=[
    Patch(color="tab:orange", label="Other result"),
    Patch(color="tab:green", label="Expected: 000 or 111"),
])
for ax in axes:
    for bars in ax.containers:
        ax.bar_label(bars, padding=3)
    ax.margins(y=0.25)
plt.show()

ValueError: No measurement results found; inspect repetition_job['raw_logs'].

**How to read the repetition-code bar plots**

- **Left — syndrome:** Each bar counts the two-bit parity result from the first ancilla at **q5**, followed by the second ancilla at **q15**. With every data qubit starting in $|+\rangle$, all four syndromes are equally likely; the dashed line marks the ideal shots/4 per syndrome.
- **Right — final data:** Each three-bit label is the final measurement of the data qubits at **q4, q10, q19**, in that order. After feedback correction, only `000` and `111` (green) are expected, about half the shots each. The title gives their combined observed fraction.

In [33]:
# Plot Repetition-code joint outcomes: syndrome versus final data
from matplotlib.patches import Rectangle

results = repetition_job["measurement_results"]
if results is None:
    raise ValueError("No measurement results found; inspect repetition_job['raw_logs'].")

results = {
    "shots": results["shots"],
    "syndrome_counts": results["register_counts"]["syndrome"],
    "data_result_counts": results["register_counts"]["result"],
    "joint_counts": {"->".join(key.split()): count for key, count in results["joint_counts"].items()},
}

syndrome_labels = ["00", "01", "10", "11"]
data_labels = [f"{value:03b}" for value in range(8)]
joint_matrix = pd.DataFrame(0, index=syndrome_labels, columns=data_labels)
for outcome, count in results["joint_counts"].items():
    syndrome, data_result = outcome.split("->")
    joint_matrix.loc[syndrome, data_result] += count

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
image = ax.imshow(joint_matrix.to_numpy(), cmap="Blues", aspect="auto", vmin=0)
ax.set_xticks(range(len(data_labels)), data_labels)
ax.set_yticks(range(len(syndrome_labels)), syndrome_labels)
ax.set(
    title="Joint outcomes (green outline: expected 000 or 111 for every syndrome)",
    xlabel="Final data (q4, q10, q19)",
    ylabel="Syndrome (ancillas q5, q15)",
)
fig.colorbar(image, ax=ax, label="Shots")
for data_result in ("000", "111"):
    column = data_labels.index(data_result)
    ax.add_patch(Rectangle(  # inset slightly so the outline is not hidden by the axes edge
        (column - 0.46, -0.46), 0.92, len(syndrome_labels) - 0.08,
        fill=False, edgecolor="limegreen", linewidth=2,
    ))
peak_count = joint_matrix.to_numpy().max()
for row, syndrome in enumerate(syndrome_labels):
    for column, data_result in enumerate(data_labels):
        count = joint_matrix.loc[syndrome, data_result]
        if count:
            ax.text(
                column, row, str(count), ha="center", va="center",
                color="white" if count > peak_count / 2 else "black",
            )
plt.show()

ValueError: No measurement results found; inspect repetition_job['raw_logs'].

**How to read the repetition-code heatmap**

Each square combines a syndrome (row: first ancilla at q5, second at q15) with a final data result (column: q4, q10, q19). Darker blue means more shots. The green outlines mark the `000` and `111` columns: whichever syndrome was measured, the correction should leave the data in one of these two. Counts outside those columns show departures from ideal correction; the plot alone cannot identify their cause.